# ch03 销售趋势预测

本 Notebook 对清洗后的产品销售数据进行销售趋势预测，包括月度聚合、简单移动平均（SMA）、指数平滑（SES）和预测误差评估。

## 1. 环境配置与数据加载

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('.').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from src.utils import (
    load_preprocessed,
    save_dataframe,
    save_markdown,
    save_figure,
    OUTPUT_BASE,
    PLT_CONFIG,
)

print('环境配置完成。')

In [ ]:
# 加载清洗后的数据
df = load_preprocessed('ch01_data_cleaning', 'product')
print(f'数据形状: {df.shape}')
print(f'数据列: {df.columns.tolist()}')
df.head()

## 2. 按月聚合销售额

In [ ]:
# 确定日期列名和销售额列名
date_col = 'date' if 'date' in df.columns else 'Date'
sales_col = 'Total_Sales_USD' if 'Total_Sales_USD' in df.columns else 'total_price'

df_copy = df.copy()
df_copy[date_col] = pd.to_datetime(df_copy[date_col])
df_copy['month'] = df_copy[date_col].dt.to_period('M')

monthly = df_copy.groupby('month')[sales_col].sum().reset_index()
monthly['month_str'] = monthly['month'].astype(str)
monthly = monthly.sort_values('month').reset_index(drop=True)

print(f'月度数据点数: {len(monthly)}')
monthly[['month_str', sales_col]]

## 3. 简单移动平均（SMA）预测

In [ ]:
# 计算移动平均
monthly['SMA_3'] = monthly[sales_col].rolling(window=3, min_periods=3).mean()
monthly['SMA_6'] = monthly[sales_col].rolling(window=6, min_periods=6).mean()

sma3_forecast = monthly['SMA_3'].iloc[-1]
sma6_forecast = monthly['SMA_6'].iloc[-1]
print(f'SMA(3) 最新值: {sma3_forecast:,.2f}')
print(f'SMA(6) 最新值: {sma6_forecast:,.2f}')

## 4. 指数平滑（SES）预测

In [ ]:
from statsmodels.tsa.holtwinters import SimpleExpSmoothing

series = monthly[sales_col].values
ses_model = SimpleExpSmoothing(series, initialization_method='estimated').fit(smoothing_level=0.3)
monthly['SES'] = ses_model.fittedvalues
ses_forecast = ses_model.forecast(1)[0]
print(f'SES 预测下一期: {ses_forecast:,.2f}')

## 5. 预测误差评估

In [ ]:
# 计算各方法的 MAE 和 RMSE
error_results = []
for method, col in [('SMA_3', 'SMA_3'), ('SMA_6', 'SMA_6'), ('SES', 'SES')]:
    valid = monthly.dropna(subset=[col])
    if len(valid) > 0:
        actual = valid[sales_col].values
        predicted = valid[col].values
        mae = np.mean(np.abs(actual - predicted))
        rmse = np.sqrt(np.mean((actual - predicted) ** 2))
        error_results.append({
            'Method': method,
            'MAE': round(mae, 2),
            'RMSE': round(rmse, 2),
            'N': len(valid)
        })
        print(f'{method}: MAE={mae:,.2f}, RMSE={rmse:,.2f} (n={len(valid)})')

error_df = pd.DataFrame(error_results)
error_df

## 6. 趋势可视化

In [ ]:
# 历史趋势 + 预测曲线图
output_dir = OUTPUT_BASE / 'sales_forecasting'
output_dir.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=PLT_CONFIG['figsize'])
x = range(len(monthly))
labels = monthly['month_str'].tolist()

ax.plot(x, monthly[sales_col], marker='o', label='实际销售额', color='#2c3e50', linewidth=2)
ax.plot(x, monthly['SMA_3'], marker='s', label='SMA(3)', color='#3498db', linewidth=1.5, linestyle='--')
ax.plot(x, monthly['SMA_6'], marker='^', label='SMA(6)', color='#e67e22', linewidth=1.5, linestyle='--')
ax.plot(x, monthly['SES'], marker='d', label='SES', color='#e74c3c', linewidth=1.5, linestyle='-.')

ax.set_xlabel('月份')
ax.set_ylabel('销售额 (USD)')
ax.set_title('月度销售额趋势与预测')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
save_figure(fig, 'sales_forecast_trend.png', output_dir, PLT_CONFIG['dpi'])
plt.show()

In [ ]:
# 预测误差对比图
if len(error_df) > 0:
    fig2, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].bar(error_df['Method'], error_df['MAE'], color=['#3498db', '#e67e22', '#e74c3c'])
    axes[0].set_title('MAE 对比')
    axes[0].set_ylabel('MAE')

    axes[1].bar(error_df['Method'], error_df['RMSE'], color=['#3498db', '#e67e22', '#e74c3c'])
    axes[1].set_title('RMSE 对比')
    axes[1].set_ylabel('RMSE')

    plt.suptitle('预测误差对比', fontsize=14)
    plt.tight_layout()
    save_figure(fig2, 'forecast_error_comparison.png', output_dir, PLT_CONFIG['dpi'])
    plt.show()

## 7. 分析总结

In [ ]:
print('=' * 50)
print('销售趋势预测 - 分析总结')
print('=' * 50)
print(f'月度数据点数: {len(monthly)} 个月')
print(f'月均销售额: ${monthly[sales_col].mean():,.0f}')
print(f'SMA(3) 预测值: ${sma3_forecast:,.0f}')
print(f'SMA(6) 预测值: ${sma6_forecast:,.0f}')
print(f'SES 预测值: ${ses_forecast:,.0f}')
print('=' * 50)
print('分析结果已保存至 outputs/sales_forecasting/')